In [1]:
import os
import sys
import torch
from pathlib import Path

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# (Optional) install deps if needed
# !pip -q install -r requirements.txt

# Move from notebooks/ to repo root so that experiments/, models/, etc. are visible.
nb_dir = Path(os.getcwd())
repo_root = nb_dir.parent
os.chdir(repo_root)
print("Changed working directory to repo root:", os.getcwd())

# Make sure repo root is on sys.path
ROOT = os.path.abspath(os.getcwd())
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

os.makedirs("results/checkpoints", exist_ok=True)
os.makedirs("results/figures", exist_ok=True)
os.makedirs("results", exist_ok=True)

print("Ready.")

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Torch: 2.10.0+cpu
CUDA available: False
Ready.


In [3]:
# Quick smoke training: all 5 models
# Uses GPU automatically if available (train.py selects cuda when possible).

import os
import sys
import subprocess
from pathlib import Path

print("CWD:", os.getcwd())
print("Python executable:", sys.executable)

# Sanity check: are we at repo root?
if not Path("experiments/train.py").is_file():
    raise FileNotFoundError(
        "Can't find experiments/train.py from current working directory. "
        "In Colab, you likely need to `%cd` into the repo root first."
    )


def run(args: list[str]):
    cmd_str = " ".join(args)
    print("\n$", cmd_str)
    r = subprocess.run(args, text=True, capture_output=True)
    if r.stdout:
        print(r.stdout)
    if r.returncode != 0:
        if r.stderr:
            print(r.stderr)
        raise RuntimeError(f"Command failed with exit code {r.returncode}: {cmd_str}")


EPOCHS = 10
SEQ_LEN = 20
BATCH_SIZE = 32
LR = 1e-3

models = ["mlp", "deep", "rnn", "lstm", "residual"]

for m in models:
    run(
        [
            sys.executable,
            "experiments/train.py",
            "--model",
            m,
            "--epochs",
            str(EPOCHS),
            "--seq-len",
            str(SEQ_LEN),
            "--batch-size",
            str(BATCH_SIZE),
            "--lr",
            str(LR),
            "--no-resume",
            "--checkpoint-path",
            f"results/checkpoints/{m}.pt",
            "--results-path",
            f"results/{m}_results.npz",
        ]
    )

print("Done training all models.")

CWD: /content
Python executable: /usr/bin/python3


FileNotFoundError: Can't find experiments/train.py from current working directory. In Colab, you likely need to `%cd` into the repo root first.

In [ ]:
# Plot ECE over time for all models

from experiments.plot_ece_over_time import plot_ece_over_time

npz_paths = [
    "results/mlp_results.npz",
    "results/deep_results.npz",
    "results/rnn_results.npz",
    "results/lstm_results.npz",
    "results/residual_results.npz",
]
labels = ["mlp", "deep", "rnn", "lstm", "residual"]

plot_ece_over_time(
    npz_paths=npz_paths,
    labels=labels,
    save_path="results/figures/ece_over_time_all.png",
)

In [ ]:
# Compact summary table for each saved .npz

import os
import numpy as np

try:
    import pandas as pd

    PANDAS_AVAILABLE = True
except Exception:
    pd = None
    PANDAS_AVAILABLE = False

runs = [
    ("mlp", "results/mlp_results.npz"),
    ("deep", "results/deep_results.npz"),
    ("rnn", "results/rnn_results.npz"),
    ("lstm", "results/lstm_results.npz"),
    ("residual", "results/residual_results.npz"),
]

rows = []
for name, path in runs:
    if not os.path.isfile(path):
        rows.append({"model": name, "path": path, "status": "missing"})
        continue

    d = np.load(path)
    row = {"model": name, "path": path, "status": "ok"}

    # Final epoch metrics (if present)
    if "val_loss" in d:
        row["final_val_loss"] = float(np.asarray(d["val_loss"])[-1])
    if "train_loss" in d:
        row["final_train_loss"] = float(np.asarray(d["train_loss"])[-1])
    if "ece_over_time" in d:
        row["final_ece"] = float(np.asarray(d["ece_over_time"])[-1])

    # Averages over the saved predictions
    if "per_example_loss" in d:
        pel = np.asarray(d["per_example_loss"]).ravel()
        row["mean_per_example_loss"] = float(np.mean(pel))

    # Classification-specific keys (may be absent for regression runs)
    if "ece" in d:
        row["ece_single"] = float(np.asarray(d["ece"]).ravel()[0])
    if "confidences" in d:
        conf = np.asarray(d["confidences"]).ravel()
        row["mean_confidence"] = float(np.mean(conf))

    rows.append(row)

if PANDAS_AVAILABLE:
    df = pd.DataFrame(rows)
    # Pretty ordering if columns exist
    preferred = [
        "model",
        "status",
        "final_val_loss",
        "final_ece",
        "final_train_loss",
        "mean_per_example_loss",
        "mean_confidence",
        "ece_single",
        "path",
    ]
    cols = [c for c in preferred if c in df.columns] + [c for c in df.columns if c not in preferred]
    display(df[cols])
else:
    # Fallback: simple text table
    for r in rows:
        print(r)